# 10 - Window Functions and Pipeline Recap

**Suggested time: 15 minutes**

A window calculates within a related group while retaining the individual rows. A common data-engineering use is choosing the latest record for each key.

## Learning objectives

By the end of this notebook, you will be able to:

- define a partition and ordering for a window;
- assign row numbers within each business key;
- select the latest record deterministically; and
- map the ten lessons to an end-to-end batch pipeline.

## Prerequisite recap

Notebook 05 warned that `dropDuplicates(['key'])` may keep an arbitrary row when duplicates differ. A window lets the business rule decide which row wins.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

order_events = spark.createDataFrame(
    [
        ('O100', 'Placed', '2026-01-05 08:00:00', 1),
        ('O100', 'Shipped', '2026-01-05 12:00:00', 2),
        ('O101', 'Placed', '2026-01-06 09:00:00', 1),
        ('O101', 'Cancelled', '2026-01-06 11:30:00', 2),
        ('O102', 'Placed', '2026-01-08 10:00:00', 1),
    ],
    'order_id STRING, status STRING, event_time_text STRING, event_sequence INT',
).withColumn(
    'event_timestamp', F.col('event_time_text').cast('timestamp')
).drop('event_time_text')

order_events.orderBy('order_id', 'event_timestamp').show(truncate=False)

## Define the window

- `partitionBy('order_id')` restarts the calculation for each order.
- `orderBy(...desc())` puts the newest event first.
- `event_sequence` breaks a timestamp tie so the result remains deterministic.

In [ ]:
latest_event_window = Window.partitionBy('order_id').orderBy(
    F.col('event_timestamp').desc(),
    F.col('event_sequence').desc(),
)

## Number rows and choose the winner

`row_number` assigns 1, 2, 3, and so on inside each ordered partition. Filtering to row 1 keeps the latest event for every order.

In [ ]:
ranked_events = order_events.withColumn(
    'event_row_number',
    F.row_number().over(latest_event_window),
)
ranked_events.orderBy('order_id', 'event_row_number').show(truncate=False)

latest_order_status = (
    ranked_events
    .filter(F.col('event_row_number') == 1)
    .drop('event_row_number')
    .orderBy('order_id')
)
latest_order_status.show(truncate=False)

The result has exactly one row per order: O100 is `Shipped`, O101 is `Cancelled`, and O102 is `Placed`.

## Your turn

Create `latest_product_prices` with one row per product. Choose the greatest `effective_timestamp`; use `price_sequence` as a descending tie-breaker. Remove the helper row-number column.

In [ ]:
product_price_history = spark.createDataFrame(
    [
        ('P001', 18.00, '2026-01-01 09:00:00', 1),
        ('P001', 18.50, '2026-01-05 09:00:00', 2),
        ('P002', 740.00, '2026-01-01 09:00:00', 1),
        ('P002', 750.00, '2026-01-06 09:00:00', 2),
        ('P003', 12.00, '2026-01-08 09:00:00', 1),
    ],
    'product_id STRING, price DOUBLE, effective_time_text STRING, price_sequence INT',
).withColumn(
    'effective_timestamp', F.col('effective_time_text').cast('timestamp')
).drop('effective_time_text')

# Write your solution here.

### Expected result

There are three rows: P001 costs 18.50, P002 costs 750.00, and P003 costs 12.00.

### Solution - reveal after attempting

In [ ]:
latest_price_window = Window.partitionBy('product_id').orderBy(
    F.col('effective_timestamp').desc(),
    F.col('price_sequence').desc(),
)

latest_product_prices = (
    product_price_history
    .withColumn('price_row_number', F.row_number().over(latest_price_window))
    .filter(F.col('price_row_number') == 1)
    .drop('price_row_number')
    .orderBy('product_id')
)
latest_product_prices.show(truncate=False)

## Afternoon pipeline checklist

A typical batch solution now follows a familiar path:

1. **Read** the three Lakehouse CSV files with explicit schemas - notebook 09.
2. **Inspect and type** each dataset - notebooks 02 and 03.
3. **Clean and derive** fields such as order value - notebooks 04, 05, and 08.
4. **Join** orders to customer and product lookups - notebook 07.
5. **Aggregate and sort** the requested business result - notebook 06.
6. **Choose a latest record** when the problem contains history - this notebook.
7. **Write and verify** a managed Delta result - notebook 09.

At each stage, check the schema, grain, row count, nulls, and duplicate keys.

## Key takeaway

Windows preserve row detail while calculating within business groups. Explicit partitioning and ordering make record selection repeatable.

You now have the core tools needed for the afternoon batch data-engineering problem.